In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import datetime

sys.path.append("..")

## Connect DB

In [ ]:
from src.setup.setup import setup
from src.config import DATABASE_PATH

setup(DATABASE_PATH.absolute().as_posix())

# Get Data

In [ ]:
import ezyquant as ez
from ezyquant.reader import _SETDataReaderCached
import ezyquant.fields as fld

In [ ]:
sdr = _SETDataReaderCached()
start_date_data = datetime.date(2019, 1, 1)  # ISO format: 2019-01-01
start_date_backtest = datetime.date(2020, 1, 1)  # ISO format: 2020-01-01
end_date = datetime.date.fromisoformat(sdr.last_update())

In [ ]:
ssc = ez.SETSignalCreator(
    start_date=start_date_data.isoformat(),
    end_date=end_date.isoformat(),
    index_list=["BANK"],
)

In [ ]:
close_df = ssc.get_data(field="close", timeframe="daily")

## Experiment

### XD dates

In [ ]:
import pandas as pd

In [ ]:
rights_benefit_df = sdr.get_rights_benefit(
    symbol_list=list(close_df.columns),
)

rights_benefit_df

In [ ]:
rights_benefit_df["symbol"].unique()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def plot_bank(symbol: str):
    bank_cash_dividends_df = rights_benefit_df[
        (rights_benefit_df["symbol"] == symbol)
        & (rights_benefit_df["ca_type"] == "CD")
    ]

    bank_close_df = close_df[symbol]

    # ---- 2.2  Grab the close price that corresponds to each dividend date ----
    #   (if a dividend date falls on a non‑trading day we forward‑fill to the next
    #    available price – change the method if you prefer 'nearest')
    bank_cash_dividends_df['price_at_sign'] = bank_cash_dividends_df['sign_date'].map(
        bank_close_df.reindex(bank_cash_dividends_df['sign_date'], method='ffill')
    )
    bank_cash_dividends_df['price_at_pay'] = bank_cash_dividends_df['pay_date'].map(
        bank_close_df.reindex(bank_cash_dividends_df['pay_date'], method='ffill')
    )

    # ---- 2.3  Plot the line ----
    fig, ax = plt.subplots(figsize=(12, 5))

    ax.plot(bank_close_df.index, bank_close_df.values,
            color='steelblue', linewidth=1.5, label='Close price')

    # ---- 2.4  Scatter‑plot the dividend dates ----
    #   Sign dates → red diamonds
    ax.scatter(bank_cash_dividends_df['sign_date'], bank_cash_dividends_df['price_at_sign'],
            color='red', s=80, marker='D', zorder=5,
            label='Sign date (dividend)')

    #   Pay dates → green triangles (optional – you can keep just one)
    ax.scatter(bank_cash_dividends_df['pay_date'], bank_cash_dividends_df['price_at_pay'],
            color='green', s=60, marker='^', zorder=5,
            label='Pay date (dividend)')

    # ---- 2.5  (Optional) Vertical lines & annotation of DPS ----
    for date, dps in zip(bank_cash_dividends_df['sign_date'], bank_cash_dividends_df['dps']):
        ax.axvline(date, color='gray', linestyle='--', linewidth=0.8)
        ax.annotate(f'{dps:.2f}', xy=(date, ax.get_ylim()[1] * 0.97),
                    fontsize=7, color='darkred', rotation=90, ha='right')

    # ---- 2.6  Beautify the axes ----
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    plt.xticks(rotation=45)

    ax.set_title(f'{symbol} – Close Price with Dividend Dates Marked')
    ax.set_xlabel('Date')
    ax.set_ylabel('Close price (THB)')
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
for symbol in rights_benefit_df["symbol"].unique():
    plot_bank(symbol)

### Valuation factors

In [ ]:
from src.strategies.valuation_factor import get_valuation_metrics
pb, pe, ev_ebitda, fcf_yield = get_valuation_metrics(ssc)

In [ ]:
macd, macd_signal, macd_histogram = ssc.ta.macd(close_df, window_slow=26, window_fast=12, window_sign=9)

In [ ]:
close_df.plot()

In [ ]:
from src.strategies.valuation_factor import get_score, get_signal

In [ ]:
score = get_score(ssc, close_df)
signal = get_signal(ssc, close_df)
# shares = get_shares(signal)

## Backtest

In [ ]:
from src.strategies.valuation_factor import get_backtest_algorithm

In [ ]:
INITIAL_CASH = 50000
PCT_COMMISSION = 0.15
PCT_SELL_SLIP = 0.01

In [ ]:
result = ez.backtest(
    signal_df=signal,
    backtest_algorithm=get_backtest_algorithm(pos_num=20),  # TODO: Remove magic number
    start_date=start_date_backtest.isoformat(),
    end_date=end_date.isoformat(),
    initial_cash=INITIAL_CASH,
    pct_commission=PCT_COMMISSION,
    pct_sell_slip=PCT_SELL_SLIP,
    price_match_mode="weighted",
    signal_delay_bar=1
)

In [ ]:
set_index_df = sdr.get_data_index_daily(
    field=fld.D_INDEX_CLOSE,
    index_list=[fld.MARKET_SET],
    start_date=start_date_data.isoformat(),
    end_date=end_date.isoformat(),
)

set_index_return = set_index_df.pct_change().fillna(0.0)

In [ ]:
result.to_basic(benchmark=set_index_return)